In [ ]:
import json
from datetime import timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from bisect import bisect_right  
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.dates as mdates
from matplotlib.colorbar import ColorbarBase
from matplotlib.colors import Normalize

import numpy as np

import sys, importlib
import obspy
from obspy import read, UTCDateTime, Stream
from obspy.core.util import AttribDict
from obspy.geodetics.base import gps2dist_azimuth
from obspy.imaging.cm import obspy_sequential
from obspy.signal.array_analysis import array_processing
from obspy.taup import TauPyModel
from obspy.geodetics import locations2degrees

In [ ]:
# --- lines to edit: your project's results ---
# Relative paths are relative to this notebook's folder (quickquake/).
DATA_ROOT = Path("../examples/pavlof_2021/data").resolve()   # the "data" folder next to your QQ_driver.py
NAMEBASE = "GAMMA"                                           # same as location_namebase in QQ_driver.py
REGION_NAME = "Pavlof"                                       # used in plot titles

# --- helpers (this notebook lives in quickquake/) ---
HELPERS_DIR = Path.cwd().resolve()
if str(HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(HELPERS_DIR))

import qq_helpers as h
importlib.reload(h)
print("helpers:", h.__file__)

# --- paths ---
CATALOG_CSV   = DATA_ROOT / f"merged/output/{NAMEBASE}_hypodd_catalog.csv"   # or any other catalog CSV
PICKS_CSV     = DATA_ROOT / f"merged/input/{NAMEBASE}_picks_cleaned.csv"
STATIONS_JSON = DATA_ROOT / f"merged/input/{NAMEBASE}_station_list.json"

INDEX_PATH = DATA_ROOT / "waveform_index.pkl"


In [ ]:
catalog_df, picks_df, stations_df = h.load_inputs(CATALOG_CSV, PICKS_CSV, STATIONS_JSON)
print("Loaded:", catalog_df.shape, picks_df.shape, stations_df.shape)

# Canonical column handles
catc = h.CatalogCols()
pkc  = h.PicksCols()
stc  = h.StationCols()

# Waveform index
if INDEX_PATH.exists():
    wf_index = h.WaveformIndex.load(INDEX_PATH)
else:
    wf_index = h.WaveformIndex.build([DATA_ROOT], glob_pattern="**/waveforms/*.mseed")
    wf_index.save(INDEX_PATH)

stream_cache = h.StreamCache()
print("Indexed mseed files:", len(wf_index.df))

print("Catalog time min/max:", catalog_df[catc.time].min(), catalog_df[catc.time].max())
print("Waveforms time min/max:",
      pd.to_datetime(wf_index.df["t0"], unit="s", utc=True).min(),
      pd.to_datetime(wf_index.df["t1"], unit="s", utc=True).max())


In [ ]:
# check 
print("Catalog time min/max:", catalog_df["datetime"].min(), catalog_df["datetime"].max())
print("Waveforms time min/max:",
      pd.to_datetime(wf_index.df["t0"], unit="s", utc=True).min(),
      pd.to_datetime(wf_index.df["t1"], unit="s", utc=True).max())


MOVEOUT PLOT

In [ ]:
from pathlib import Path

for row_idx, row_event in catalog_df.iloc[0:70].iterrows():

    fig, ax, meta = h.plot_event_moveout_pretty(
        row_event=row_event,
        picks=picks_df,
        stations=stations_df,
        cache=stream_cache,
        wf_index=wf_index,

        # waveform window (seconds before / after origin time)
        pre_s=0,
        post_s=40,

        # bandpass filter (Hz)
        freqmin=1.0,
        freqmax=5.0,

        # figure size (inches)
        figsize=(10, 7),

        # vertical spacing and wiggle amplitude
        wiggle_frac_of_spacing=2.0,
        y_pad_km=2.5,
        y_tick_step_km=4.0,

        # picks
        p_color="black",
        s_color="red",
        line_height_km=2.4,
        line_width=2.0,

        # text sizes
        label_fontsize=18,
        tick_fontsize=20,
        title_fontsize=17,
        legend_fontsize=15,

        # x-axis tick spacing (seconds)
        x_tick_seconds=10,

        # optional: save each figure
        # save_path=f"moveout_event_{row_idx}.svg",
    )

    print(
        row_idx,
        "files:", [Path(f).name for f in meta["files"]],
        "stations:", len(meta["stations_used"])
    )

SPECTROGRAM

In [ ]:
for row_idx, row_event in catalog_df.iloc[0:40].iterrows():
    fig, axes, meta = h.plot_event_spectrograms_vertical_picks_only(
        row_event=row_event,
        picks=picks_df,
        stations=stations_df,
        cache=stream_cache,
        wf_index=wf_index,
        pre_s=30,
        post_s=90,
        window_s=2,
        overlap_frac=0.8,
        freq_min=0,
        freq_max=20,
    )
    print("files:", [Path(f).name for f in meta["files"]])
    print("stations used:", len(meta["stations_used"]))


In [ ]:
start_2021 = "2020-01-01T00:00:00Z"
end_2021   = "2023-12-31T23:59:59Z"

h.plot_mapview_seismicity(
    catalog=catalog_df,
    stations=stations_df,
    start=start_2021,
    end=end_2021,
    relief_zoom=12,
    savefig=False,
    ellipse_lon=-161.89, ellipse_lat=55.42,
    a_km=12, b_km=6, theta_deg=45,
    region_name=REGION_NAME,
)

In [ ]:
import matplotlib.pyplot as plt
import qq_helpers as h

fig, ax, meta = h.plot_mapview3d_months(
    catalog=catalog_df,          # the same catalog as above
    start="2021-01-01",
    end="2022-01-01",            # end is exclusive
    depth_max=40,
    exagg=4,
    view_elev=4,
    view_azim=34,
    dem_res="15s",
    savefig=False,               # figures are not saved by default
)

plt.show()
print("Number of plotted seismic events:", meta["n_events"])

In [ ]:
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"   # shows the figure in VS Code and in Jupyter

fig = h.plot_catalog_3d_topo_seismicity(
    catalog=catalog_df,
    mode="time",
    show=True,
    region_name=REGION_NAME,
)



In [ ]:
fig = h.plot_catalog_3d_topo_seismicity(
    catalog=catalog_df,
    mode="year_month",
    year=2021,
    show=True,
    region_name=REGION_NAME,
)


In [ ]:
# 1) Edit these periods for your region / year
event_spans = [
    ("2021-08-05", "2021-08-06", "Ash emissions"),
    ("2021-08-24", "2021-08-26", "Ash emissions"),
    ("2021-09-01", "2021-10-04", "Ash emissions"),
    ("2021-10-14", "2021-11-10", "Ash emissions"),
    ("2021-12-03", "2021-12-05", "Ash emissions"),
    ("2021-11-08", "2021-12-31", "Intermittent lahars"),
    ("2021-11-08", "2021-12-31", "Intermittent lava flows"),
]

# 2) One plotting call (does NOT save a PNG by default)
fig, ax, meta = h.plot_timeline_histogram(
    catalog=catalog_df,         # or another catalog, e.g. events inside an ellipse
    start=start_2021,
    end=end_2021,
    spans=event_spans,
    n_bins=90,
    savefig=False,              # set True to save the figure
)
plt.show()